# LociSimiles Word2Vec Example

This notebook shows how to run the Burns-style Word2Vec retrieval pipeline.

## Requirements

Install optional support if needed:

```bash
pip install "locisimiles[word2vec]"
```

This notebook loads the pretrained model saved in this repo at:
`./models/pretrained_glove_wiki_gigaword_50.model`

In [1]:
from pathlib import Path

from locisimiles.document import Document
from locisimiles.evaluator import IntertextEvaluator
from locisimiles.pipeline import (
    Word2VecRetrievalPipeline,
    pretty_print,
)

/Users/julianschelb/.pyenv/versions/locisimiles-test/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
query_doc = Document("./hieronymus_samples.csv", author="Hieronymus")
source_doc = Document("./vergil_samples.csv", author="Vergil")

print(f"Query segments: {len(query_doc)}")
print(f"Source segments: {len(source_doc)}")

Query segments: 11
Source segments: 10


In [3]:
# Load the pretrained Word2Vec model saved in the models folder.
model_path = Path("./models/pretrained_glove_wiki_gigaword_50.model")

if not model_path.exists():
    raise FileNotFoundError(
        f"Expected model file not found: {model_path.resolve()}"
    )

print(f"Using local model: {model_path.resolve()}")

Using local model: /Users/julianschelb/Repositories/locisimiles/examples/models/pretrained_glove_wiki_gigaword_50.model


In [4]:
pipeline_word2vec = Word2VecRetrievalPipeline(
    model_path=model_path,
    top_k=10,
    similarity_threshold=0.85,
    interval=2,
    order_free=True,
)

results_word2vec = pipeline_word2vec.run(
    query=query_doc,
    source=source_doc,
    top_k=10,
)

pretty_print(results_word2vec)


▶ Query segment 'hier. adv. iovin. 1.1':
  verg. aen. 10.636          candidate=+1.000  judgment=1.000
  verg. georg. 1.197         candidate=+0.844  judgment=0.000
  verg. georg. 2.475         candidate=+0.821  judgment=0.000
  verg. aen. 1.177           candidate=+0.820  judgment=0.000
  verg. aen. 10.875          candidate=+0.764  judgment=0.000
  verg. aen. 11.508          candidate=+0.736  judgment=0.000
  verg. aen. 4.172           candidate=+0.727  judgment=0.000
  verg. ecl. 8.62            candidate=+0.715  judgment=0.000
  verg. ecl. 3.26            candidate=+0.673  judgment=0.000
  verg. ecl. 3.49            candidate=+0.000  judgment=0.000

▶ Query segment 'hier. adv. iovin. 1.41':
  verg. aen. 11.508          candidate=+1.000  judgment=1.000
  verg. georg. 1.197         candidate=+0.791  judgment=0.000
  verg. aen. 10.875          candidate=+0.767  judgment=0.000
  verg. aen. 1.177           candidate=+0.746  judgment=0.000
  verg. georg. 2.475         candidate=+0.744  

In [5]:
evaluator = IntertextEvaluator(
    query_doc=query_doc,
    source_doc=source_doc,
    ground_truth_csv="./ground_truth.csv",
    pipeline=pipeline_word2vec,
    top_k=10,
    threshold=0.85,
)

print("Macro:")
print(evaluator.evaluate(average="macro", with_match_only=True))
print("\nMicro:")
print(evaluator.evaluate(average="micro", with_match_only=True))

Macro:
   precision  recall        f1  accuracy   fpr   fnr   smr  tp  fp  fn  tn
0   0.436667     0.9  0.541905      0.81  0.18  0.01  0.19   9  18   1  72

Micro:
   precision  recall        f1  accuracy   fpr   fnr   smr  tp  fp  fn  tn
0   0.333333     0.9  0.486486      0.81  0.18  0.01  0.19   9  18   1  72


In [6]:
pipeline_word2vec.to_csv("./results_word2vec.csv", results=results_word2vec)
pipeline_word2vec.to_json("./results_word2vec.json", results=results_word2vec)
print("Saved results_word2vec.csv and results_word2vec.json")

Saved results_word2vec.csv and results_word2vec.json
